In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "sumy", "bert-score", "sacrebleu"], check=True)

import os
import math
import numpy as np
import pandas as pd
import torch
import chardet
import evaluate
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")  # для Colab без GUI

# sumy — для экстрактивных методов
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer as SumyTokenizer
from sumy.summarizers.text_rank import TextRankSummarizer
from sumy.summarizers.lex_rank import LexRankSummarizer   # PageRank на графе предложений
from sumy.summarizers.lsa import LsaSummarizer
from sumy.nlp.stemmers import Stemmer as SumyStemmer
from sumy.utils import get_stop_words

from datasets import Dataset, DatasetDict, concatenate_datasets, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

In [ ]:
# @title 2. Загрузка и предобработка датасета
import os
from datasets import Dataset, DatasetDict

# Клонируем репозиторий
!git clone https://github.com/iis-research-team/summarization-dataset.git
data_path = "summarization-dataset/dataset"

# Собираем все тексты и абстракты
texts = []
abstracts = []

# Проходим по всем папкам научных статей
for domain in os.listdir(data_path):
    domain_path = os.path.join(data_path, domain)
    if os.path.isdir(domain_path):
        for paper_folder in os.listdir(domain_path):
            paper_path = os.path.join(domain_path, paper_folder)
            text_file = os.path.join(paper_path, "text.txt")
            abstract_file = os.path.join(paper_path, "abstract.txt")

            if os.path.exists(text_file) and os.path.exists(abstract_file):
                with open(text_file, 'r', encoding='utf-8') as f:
                    text = f.read().strip()
                with open(abstract_file, 'r', encoding='utf-8') as f:
                    abstract = f.read().strip()

                # Добавляем только непустые пары
                if text and abstract:
                    texts.append(text)
                    abstracts.append(abstract)

print(f"Загружено {len(texts)} пар статья-аннотация.")

# Создаем датасет Hugging Face
dataset = Dataset.from_dict({"text": texts, "summary": abstracts})

# Разделяем на train (90%) и validation (10%)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset = DatasetDict({
    'train': dataset['train'],
    'validation': dataset['test']
})

print(dataset)